# Self-host LLM on Colab → Public OpenAI-compatible API (GGUF / llama.cpp-style)

Notebook này chạy **GGUF model** theo kiểu **KoboldCpp/llama.cpp** và expose ra internet qua **OpenAI-compatible API**.

- Khuyến nghị: **ngrok** (ổn định hơn cho đồ án/demo).

- Tuỳ chọn: **TryCloudflare** (quick tunnel) có thể chập chờn/DNS không ổn định.



Cuối notebook sẽ in ra:

- `SELFHOST_BASE_URL = <PUBLIC_URL>/v1`

- `SELFHOST_MODEL = <MODEL_ID>`

Bạn copy 2 giá trị đó vào project của bạn.


## 0) Cấu hình

- Chọn model GGUF bằng `KCPP_MODEL_URL` (URL trực tiếp tới file `.gguf` trên HuggingFace).

- Context: `KCPP_CONTEXT` (vd `4096`).

- GPU layers: `KCPP_GPU_LAYERS` (tăng nếu VRAM đủ; giảm nếu bị OOM).

- Khuyến nghị: điền `NGROK_AUTHTOKEN` để dùng ngrok (ổn định).

- Nếu bạn vẫn muốn thử TryCloudflare: set `USE_TRYCLOUDFLARE=1`.

Ghi chú:

- KoboldCpp trả về model id qua `/v1/models`; notebook sẽ tự đọc và set `MODEL_ID`.

- Colab đôi khi không resolve được domain `*.trycloudflare.com`, nên PUBLIC test có thể fail ngay trên Colab dù URL vẫn chạy từ máy local.

In [ ]:

# ====== CONFIG (KoboldCpp) ======

import os

# URL trực tiếp tới file .gguf (KoboldCpp hỗ trợ model URL)
os.environ["KCPP_MODEL_URL"] = os.environ.get(
    "KCPP_MODEL_URL",
    "https://huggingface.co/concedo/KobbleTinyV2-1.1B-GGUF/resolve/main/KobbleTiny-Q4_K.gguf",
 )

# Context length
os.environ["KCPP_CONTEXT"] = os.environ.get("KCPP_CONTEXT", "4096")

# GPU layers (tăng nếu VRAM đủ; giảm nếu OOM)
os.environ["KCPP_GPU_LAYERS"] = os.environ.get("KCPP_GPU_LAYERS", "99")

# Dùng CUDA nếu có GPU (Colab thường là T4)
os.environ["KCPP_USECUDA"] = os.environ.get("KCPP_USECUDA", "1")

# Ưu tiên ngrok (ổn định). Nếu bạn muốn thử TryCloudflare thì set biến này = 1.
os.environ["USE_TRYCLOUDFLARE"] = os.environ.get("USE_TRYCLOUDFLARE", "0")

# (Tuỳ chọn) Nếu TryCloudflare bị lỗi, notebook sẽ dùng ngrok nếu bạn set token này.
os.environ["NGROK_AUTHTOKEN"] = os.environ.get("NGROK_AUTHTOKEN", "")  # <-- điền token nếu muốn

# API key "dummy" (OpenAI client yêu cầu có api_key; server thường không check)
os.environ["API_KEY"] = os.environ.get("API_KEY", "dummy")

print("KCPP_MODEL_URL =", os.environ["KCPP_MODEL_URL"])
print("KCPP_CONTEXT =", os.environ["KCPP_CONTEXT"])
print("KCPP_GPU_LAYERS =", os.environ["KCPP_GPU_LAYERS"])
print("KCPP_USECUDA =", os.environ["KCPP_USECUDA"])
print("USE_TRYCLOUDFLARE =", os.environ["USE_TRYCLOUDFLARE"])
print("NGROK_AUTHTOKEN set?", bool(os.environ["NGROK_AUTHTOKEN"]))


## 1) Cài dependencies + tải KoboldCpp

Cell này sẽ:

- Cài Python packages (OpenAI client, requests, ngrok)

- Tải binary `koboldcpp` (Linux)

Sau đó ta sẽ chạy server KoboldCpp (OpenAI-compatible `/v1`) ở `http://127.0.0.1:5001/v1`.

In [ ]:

!pip -q install --upgrade pip

!pip -q install "openai>=1.0.0" requests pyngrok nest_asyncio

import pathlib, subprocess

# Download KoboldCpp Linux binary (official colab uses this mirror)
KCPP_BIN = pathlib.Path("/content/koboldcpp")
if not KCPP_BIN.exists():
    subprocess.run("rm -f /content/koboldcpp", shell=True, check=False)
    subprocess.run("wget -q -O /content/koboldcpp https://kcpplinux.concedo.workers.dev", shell=True, check=True)
    subprocess.run("chmod +x /content/koboldcpp", shell=True, check=True)

print("KoboldCpp binary:", KCPP_BIN, "exists?", KCPP_BIN.exists())
!/content/koboldcpp --help | head -n 25


## 2) Start KoboldCpp OpenAI API server (background)

Server sẽ chạy ở `http://127.0.0.1:5001/v1`.

KoboldCpp có OpenAI-compatible API sẵn ở route `/v1` (bao gồm `/v1/models`, `/v1/chat/completions`).

In [ ]:

import os, subprocess, pathlib, shlex

MODEL_URL = os.environ["KCPP_MODEL_URL"]
CTX = int(os.environ.get("KCPP_CONTEXT", "4096"))
GPU_LAYERS = int(os.environ.get("KCPP_GPU_LAYERS", "99"))
USECUDA = os.environ.get("KCPP_USECUDA", "1").strip().lower() in ("1", "true", "yes")

# Dọn log cũ
for f in ["koboldcpp.log", "tunnel.log"]:
    try:
        pathlib.Path(f).unlink()
    except FileNotFoundError:
        pass

# Kill process cũ (nếu chạy lại cell)
subprocess.run("pkill -f 'koboldcpp' >/dev/null 2>&1 || true", shell=True)
subprocess.run("pkill -f 'koboldcpp_linux' >/dev/null 2>&1 || true", shell=True)

# Start server (default port 5001)
base = "/content/koboldcpp"
cmd_parts = [
    "nohup", base,
    "--model", MODEL_URL,
    "--gpulayers", str(GPU_LAYERS),
    "--contextsize", str(CTX),
    "--chatcompletionsadapter", "AutoGuess",
    "--quiet",
 ]

# Enable CUDA like official colab: --usecuda 0 mmq
if USECUDA:
    cmd_parts += ["--usecuda", "0", "mmq"]

cmd = " ".join(shlex.quote(p) for p in cmd_parts) + " > koboldcpp.log 2>&1 &"
print(cmd)
subprocess.run(cmd, shell=True, check=True)
print("Started KoboldCpp server. Waiting for readiness...")


## 3) Health check: local `/v1/models`

Cell này đợi server sẵn sàng. Nếu fail, xem `koboldcpp.log` để biết lỗi load model/VRAM.

In [ ]:

import time, requests, json, os

LOCAL_BASE = "http://127.0.0.1:5001/v1"

def wait_for_local(timeout_s=300):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            r = requests.get(LOCAL_BASE + "/models", timeout=5)
            if r.status_code == 200:
                return r.json()
        except Exception:
            pass
        time.sleep(2)
    return None

data = wait_for_local()
if not data:
    print("❌ Server chưa sẵn sàng. In 160 dòng cuối koboldcpp.log để debug:\n")
    !tail -n 160 koboldcpp.log
    raise RuntimeError("KoboldCpp server not ready")
else:
    print("✅ Local server OK. /v1/models:")
    print(json.dumps(data, indent=2)[:1600])

    # Tự chọn MODEL_ID từ /v1/models để dùng cho OpenAI client và project
    try:
        model_id = data["data"][0]["id"]
        os.environ["MODEL_ID"] = model_id
        print("\n✅ SELFHOST_MODEL =", model_id)
    except Exception:
        print("\n⚠️ Không parse được model id từ /v1/models. Bạn có thể tự chọn model string.")


## 4) Expose public URL (TryCloudflare → fallback ngrok nếu cần)

Notebook sẽ:

1) start cloudflared tunnel

2) parse URL `https://xxxx.trycloudflare.com`

3) test DNS + test `/v1/models`

4) nếu fail → dùng ngrok nếu có `NGROK_AUTHTOKEN`


In [ ]:
import os, re, time, subprocess, pathlib, socket, requests
from pyngrok import ngrok

USE_TRYCLOUDFLARE = os.environ.get("USE_TRYCLOUDFLARE", "0").strip().lower() in ("1", "true", "yes")

def _kill_old_cloudflared() -> None:
    # Best-effort cleanup so we don't parse an old URL from a previous run.
    subprocess.run("pkill -f 'cloudflared tunnel' >/dev/null 2>&1 || true", shell=True)

def start_cloudflared() -> None:
    if not pathlib.Path("cloudflared").exists():
        subprocess.run(
            "wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            shell=True,
            check=True,
        )
        subprocess.run("chmod +x cloudflared", shell=True, check=True)

    _kill_old_cloudflared()
    pathlib.Path("tunnel.log").unlink(missing_ok=True)

    # Quick tunnel (TryCloudflare). Note: sometimes Colab cannot resolve the domain,
    # but the URL may still work from your local machine.
    subprocess.run(
        "nohup ./cloudflared tunnel --url http://127.0.0.1:5001 --no-autoupdate > tunnel.log 2>&1 &",
        shell=True,
        check=True,
    )

def parse_trycloudflare_url(timeout_s: int = 75):
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if pathlib.Path("tunnel.log").exists():
            txt = pathlib.Path("tunnel.log").read_text(errors="ignore")
            m = pattern.search(txt)
            if m:
                return m.group(0)
        time.sleep(1)
    return None

def dns_ok(url: str) -> bool:
    host = url.replace("https://", "").replace("http://", "").split("/")[0]
    try:
        socket.gethostbyname(host)
        return True
    except Exception:
        return False

def public_models_ok(public_url: str) -> bool:
    try:
        r = requests.get(public_url.rstrip("/") + "/v1/models", timeout=15)
        return r.status_code == 200
    except Exception:
        return False

public_url = None

token = os.environ.get("NGROK_AUTHTOKEN", "").strip()

# 1) Ưu tiên ngrok nếu có token
if token:
    print("Using ngrok (recommended)...")
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(5001, "http").public_url
    print("ngrok URL =", public_url)
    try:
        r = requests.get(public_url.rstrip("/") + "/v1/models", timeout=20)
        print("ngrok /v1/models status:", r.status_code)
        if r.status_code != 200:
            print(r.text[:400])
            public_url = None
    except Exception as e:
        print("⚠️ ngrok health-check failed:", e)
        public_url = None

# 2) TryCloudflare (chỉ khi được bật, hoặc khi không có token)
if (not public_url) and (USE_TRYCLOUDFLARE or (not token)):
    print("Starting TryCloudflare tunnel...")
    start_cloudflared()
    tc_url = parse_trycloudflare_url()

    if tc_url:
        print("TryCloudflare URL =", tc_url)

        # Important: do NOT fail hard just because Colab DNS can't resolve it.
        # Colab DNS failures are common; the URL can still work externally.
        if not dns_ok(tc_url):
            print("⚠️ Colab DNS không resolve được TryCloudflare domain.")
            print("   → Bạn vẫn có thể thử URL này từ máy local.")

        # If we can validate from Colab, great. If not, keep the URL anyway.
        if public_models_ok(tc_url):
            public_url = tc_url
            print("✅ TryCloudflare /v1/models OK (from Colab)")
        else:
            print("⚠️ Colab không kiểm tra được /v1/models qua TryCloudflare (DNS/tunnel/origin có thể chập chờn).")
            print("   → Giữ URL để bạn test từ local. Nếu cần ổn định hơn, dùng ngrok với NGROK_AUTHTOKEN.")
            public_url = tc_url
    else:
        print("⚠️ Không lấy được TryCloudflare URL trong tunnel.log.")

if not public_url:
    print("\n❌ Không có public URL (không lấy được TryCloudflare URL và ngrok token trống).")
    print("→ Hãy điền NGROK_AUTHTOKEN ở Cell 0 (CONFIG) rồi chạy lại cell này để dùng ngrok.")
    print("\n--- tail tunnel.log ---")
    !tail -n 120 tunnel.log
    raise RuntimeError("No public URL")

SELFHOST_BASE_URL = public_url.rstrip("/") + "/v1"
print("\n✅ SELFHOST_BASE_URL =", SELFHOST_BASE_URL)


## 5) Test chat completion (LOCAL & PUBLIC)

Nếu local OK mà public fail: lỗi tunnel/DNS.


In [ ]:

import os
import socket
from openai import OpenAI

MODEL_ID = os.environ.get("MODEL_ID", "")
API_KEY = os.environ.get("API_KEY", "dummy")

if not MODEL_ID:
    raise RuntimeError("MODEL_ID chưa được set. Hãy chạy cell Health check để lấy SELFHOST_MODEL từ /v1/models.")

client_local = OpenAI(base_url="http://127.0.0.1:5001/v1", api_key=API_KEY)
resp = client_local.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "Ping local: trả lời 1 câu ngắn."}],
    temperature=0.0,
 )
print("LOCAL:", resp.choices[0].message.content)

# Public call: TryCloudflare có thể không resolve được từ chính Colab.
# Nếu PUBLIC fail ở Colab, bạn vẫn có thể test URL từ máy local.
try:
    client_public = OpenAI(base_url=SELFHOST_BASE_URL, api_key=API_KEY)
    resp2 = client_public.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": "Ping public: trả lời 1 câu ngắn."}],
        temperature=0.0,
    )
    print("PUBLIC:", resp2.choices[0].message.content)
except Exception as e:
    print("⚠️ PUBLIC ping failed from Colab:", repr(e))
    try:
        host = SELFHOST_BASE_URL.replace("https://", "").replace("http://", "").split("/")[0].split(":")[0]
        print("PUBLIC host =", host)
        print("Colab DNS resolve =", socket.gethostbyname(host))
    except Exception as dns_e:
        print("Colab DNS resolve failed:", repr(dns_e))
    print("→ Hãy test từ máy local (curl/OpenAI client) với SELFHOST_BASE_URL ở trên.")


## 6) Copy sang project

Trong project trên máy bạn, set:

```env

LLM_PROVIDER=selfhost

SELFHOST_BASE_URL=<SELFHOST_BASE_URL ở trên>

SELFHOST_MODEL=<SELFHOST_MODEL ở Cell 3>

SELFHOST_API_KEY=dummy

```
